In [ ]:
!pip install kagglehub datasets

In [ ]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 20.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12

In [ ]:
!pip install tqdm

In [ ]:
# prompt: mount to drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from transformers import PreTrainedTokenizerFast
import os
import pandas as pd
import json
import shutil
from datasets import load_dataset

In [ ]:
import pandas as pd
from datasets import load_dataset, Dataset
from pathlib import Path
from tqdm import tqdm

In [ ]:
# === ✅ Step 1: Load Existing Merged Tokenizer ===
EXISTING_TOKENIZER_PATH = "/content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2_MERGED"
EXISTING_TOKENIZER_JSON = os.path.join(EXISTING_TOKENIZER_PATH, "tokenizer.json")
existing_tokenizer = Tokenizer.from_file(EXISTING_TOKENIZER_JSON)

In [ ]:
ds_sharegpt = load_dataset("FreedomIntelligence/sharegpt-arabic", split="train")

In [ ]:
print("✅ ShareGPT-Arabic Samples:", len(ds_sharegpt))
print("📎 Columns:", ds_sharegpt.column_names)

✅ ShareGPT-Arabic Samples: 5231
📎 Columns: ['conversations', 'lang', 'id']


In [ ]:
def convert_conversation_to_instruction(sample):
    messages = sample.get("conversations", [])

    # Ensure there are at least one human and one gpt message
    human_messages = [m for m in messages if m["from"] == "human"]
    gpt_messages = [m for m in messages if m["from"] == "gpt"]
    if not human_messages or not gpt_messages:
        return None

    instruction = human_messages[0]["value"].strip()

    # Collect follow-up human messages (after the first) as "input"
    input_parts = [m["value"].strip() for m in messages if m["from"] == "human"][1:]
    input_text = "\n".join(input_parts) if input_parts else ""

    # Take the first assistant reply as "output"
    output = gpt_messages[0]["value"].strip()

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output
    }

In [ ]:
# Apply mapping safely and remove empty rows
converted_dataset = ds_sharegpt.map(convert_conversation_to_instruction)
converted_dataset = converted_dataset.filter(lambda x: x is not None and x["output"].strip() != "")

Map:   0%|          | 0/5231 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5229 [00:00<?, ? examples/s]

In [ ]:
# Check result
print(f"✅ Total formatted samples: {len(converted_dataset)}")
print("📌 Sample:", converted_dataset[0])

✅ Total formatted samples: 5229
📌 Sample: {'conversations': [{'from': 'human', 'value': 'ما هو تدفق الجيت؟'}, {'from': 'gpt', 'value': 'Git flow هو نموذج فرع لـ Git، وهو نظام تحكم الإصدارات شائع. تم تصميم نموذج Git flow لمساعدة المطورين على العمل في مشاريع كبيرة ومعقدة عن طريق تحديد قواعد صارمة للفروع والدمج. الفكرة الأساسية وراء Git flow هي أن هناك فرعين رئيسيين في مستودع Git: فرع الـ `master`، الذي يمثل الشفرة الجاهزة للإنتاج، وفرع الـ `develop` الذي يحتوي على آخر التغييرات التطويرية. عندما يريد المطور البدء في العمل على ميزة جديدة، ينشئ فرعًا جديدًا يفرع من فرع الـ `develop`، وعندما يكون جاهزًا، يدمج تغييراته مرة أخرى في فرع الـ `develop`. يتيح هذا للمطورين المتعددين العمل على ميزات مختلفة بشكل متزامن دون التداخل في عمل بعضهم البعض. عندما يكون الشيفرة في فرع الـ `develop` جاهزة للإصدار، يتم دمجها في فرع الـ `master` ويتم وضع تسمية الإصدار عليها.'}, {'from': 'human', 'value': 'يمكنك أن تعطيني رسمًا مبنيًا على نص يوضح تدفق Git بشرط أن يحتوي على شرطات وشرطتين؟'}, {'from': 'gpt', 'value

In [ ]:
print("📎 Columns:", converted_dataset.column_names)

📎 Columns: ['conversations', 'lang', 'id', 'instruction', 'input', 'output']


In [ ]:
converted_dataset.to_json("/content/drive/MyDrive/sharegpt_arabic_instruction.jsonl", orient="records", lines=True, force_ascii=False)

Creating json from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

49801457

In [ ]:
DATA_FILES = "/content/drive/MyDrive/FINAL_GRAD_PROJ/cleaned_datasets"

In [ ]:
COLUMNS = ["instruction", "output"]  # We'll train on both
VOCAB_SIZE = 32000
OUTPUT_DIR = "/content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_merged"
TOKENIZER_JSON = "tokenizer_arabic_merged.json"

In [ ]:
all_texts = []
sample_count = {}

for file in os.listdir(DATA_FILES):
    path = os.path.join(DATA_FILES, file)
    local_samples = 0
    try:
        if file.endswith(".csv"):
            df = pd.read_csv(path)
            for _, row in df.iterrows():
                inst = str(row.get("instruction", "")).strip()
                out = str(row.get("output", "")).strip()
                if inst and out:
                    all_texts.append(inst)
                    all_texts.append(out)
                    local_samples += 1
        elif file.endswith(".jsonl"):
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        item = json.loads(line)
                        inst = str(item.get("instruction", "")).strip()
                        out = str(item.get("output", "")).strip()
                        if inst and out:
                            all_texts.append(inst)
                            all_texts.append(out)
                            local_samples += 1
                    except json.JSONDecodeError as e:
                        print(f"❌ JSONL decode error in {file}: {e}")
        elif file.endswith(".json"):
            with open(path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    for item in data:
                        inst = str(item.get("instruction", "")).strip()
                        out = str(item.get("output", "")).strip()
                        if inst and out:
                            all_texts.append(inst)
                            all_texts.append(out)
                            local_samples += 1
                except json.JSONDecodeError as e:
                    print(f"❌ JSON decode error in {file}: {e}")
        sample_count[file] = local_samples
    except Exception as e:
        print(f"⚠️ Error processing {file}: {e}")

In [ ]:
# Summary
print("\n📊 Per-file sample counts:")
total = 0
for f, count in sample_count.items():
    print(f"  ✅ {f}: {count} instruction/output pairs")
    total += count

print(f"\n🔢 Total extracted samples: {total}")
print(f"🧪 Sample: {all_texts[0]}")


📊 Per-file sample counts:
  ✅ alpaca_arabic_clean.csv: 52002 instruction/output pairs
  ✅ cleaned_beetlware.csv: 1186 instruction/output pairs
  ✅ sharegpt_arabic_instruction.jsonl: 5229 instruction/output pairs
  ✅ Copy of gemma2_ar_instruct_cleaned.jsonl: 16157 instruction/output pairs

🔢 Total extracted samples: 74574
🧪 Sample: أعط ثلاث نصائح للحفاظ على الصحة.


In [ ]:
# === ✅ Train Tokenizer ===
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

In [ ]:
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]", "<s>", "</s>"]
)

tokenizer.train_from_iterator(all_texts, trainer=trainer)

In [ ]:
tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    pair="<s> $A </s> </s> $B </s>",
    special_tokens=[
        ("<s>", tokenizer.token_to_id("<s>")),
        ("</s>", tokenizer.token_to_id("</s>"))
    ]
)

In [ ]:
# Save the tokenizer
save_path = "/content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_hf"
Path(save_path).mkdir(parents=True, exist_ok=True)

In [ ]:
tokenizer.save(f"{save_path}/tokenizer.json")

print(f"✅ Tokenizer trained and saved to {save_path}")

✅ Tokenizer trained and saved to /content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_hf


In [ ]:
OUTPUT_DIR = "/content/drive/MyDrive/FINAL_GRAD_PROJ/PRE_FINAL_TOKENIZER"

In [ ]:
from transformers import PreTrainedTokenizerFast

# Path to your trained tokenizer.json
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="/content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_hf/tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
)

In [ ]:
# Save in full HF-compatible format (tokenizer_config.json, vocab.json, etc.)
tokenizer.save_pretrained("/content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_hf")

('/content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_hf/tokenizer_config.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_hf/special_tokens_map.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/final_tokenizer_hf/tokenizer.json')

In [ ]:
# ✅ Quick test
sample = "ما هو تعريف الذكاء الاصطناعي؟"
print("🧪 Tokens:", tokenizer.tokenize(sample))
print("🧱 Token IDs:", tokenizer.encode(sample))
print("🔁 Decoded:", tokenizer.decode(tokenizer.encode(sample)))

🧪 Tokens: ['ما', 'هو', 'تعريف', 'الذكاء', 'الاصطناعي', '؟']
🧱 Token IDs: [5, 773, 834, 2856, 1245, 1194, 256, 6]
🔁 Decoded: <s> ما هو تعريف الذكاء الاصطناعي ؟ </s>
